# 02 - Exploratory Data Analysis (EDA)
## NYC Taxi Revenue Optimization Project

**Goal:** Understand the data deeply before writing a single cleaning rule.
Every cleaning decision in `03_data_cleaning.sql` will be justified by findings here.

**Source Table:** `workspace.nyc_taxi.yellow_trips_raw`  
**Total Records:** 248M trips across 2019, 2022–2025

**EDA Structure:**
1. Row counts and completeness
2. Financial column distributions (fare, tip, total)
3. Trip distance and duration distributions
4. Temporal patterns (hour, day, month)
5. Zone patterns (pickup/dropoff locations)
6. Outlier detection
7. Year-over-year comparisons

---
> All findings feed directly into cleaning thresholds and analysis strategy.


### Section 1 — Row Counts and Completeness

First thing we always check:
- How many rows per year and month
- Which columns have nulls and how many
- Are nulls random or concentrated in specific years

This tells us the overall health of the dataset before we look at values.

In [0]:
SELECT
    COUNT(*)                                            AS total_rows,
    SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END)              AS null_vendor_id,
    SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END)  AS null_pickup_dt,
    SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_dt,
    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END)          AS null_pu_location,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END)          AS null_do_location,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END)       AS null_passengers,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END)         AS null_distance,
    SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END)           AS null_fare,
    SUM(CASE WHEN tip_amount IS NULL THEN 1 ELSE 0 END)            AS null_tip,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END)          AS null_total,
    SUM(CASE WHEN RatecodeID IS NULL THEN 1 ELSE 0 END)            AS null_ratecode,
    SUM(CASE WHEN congestion_surcharge IS NULL THEN 1 ELSE 0 END)  AS null_congestion,
    SUM(CASE WHEN airport_fee IS NULL THEN 1 ELSE 0 END)           AS null_airport_fee,
    SUM(CASE WHEN cbd_congestion_fee IS NULL THEN 1 ELSE 0 END)    AS null_cbd_fee
FROM workspace.nyc_taxi.yellow_trips_raw;

### Section 2 — Confirm Nulls Are Year-Specific

The aggregate null counts look manageable, but we need to confirm
they are concentrated in specific years — not spread randomly.

This matters because:
- Congestion surcharge only launched in 2019 Q4
- CBD congestion fee only launched January 2025
- If nulls are concentrated in expected years, they are structural not errors

In [0]:
SELECT
    data_year,
    COUNT(*)                                                                                    AS total_rows,
    ROUND(SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)      AS pct_null_passengers,
    ROUND(SUM(CASE WHEN RatecodeID IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)           AS pct_null_ratecode,
    ROUND(SUM(CASE WHEN congestion_surcharge IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_null_congestion,
    ROUND(SUM(CASE WHEN airport_fee IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)          AS pct_null_airport_fee,
    ROUND(SUM(CASE WHEN cbd_congestion_fee IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)   AS pct_null_cbd_fee
FROM workspace.nyc_taxi.yellow_trips_raw
GROUP BY data_year
ORDER BY data_year;

### Section 3 — Identify Which Vendor Is Causing Null Explosion in 2025

passenger_count and RatecodeID nulls jumped from 10% in 2024 to 23% in 2025.
This is almost certainly one vendor that stopped reporting these fields.

VendorID meanings:
- 1 = Creative Mobile Technologies (CMT)
- 2 = VeriFone Inc.
- NULL = unknown

We need to find which vendor is responsible before we decide
whether to exclude their trips or just treat nulls as unknown.

In [0]:
SELECT
    data_year,
    VendorID,
    COUNT(*)                                                                               AS total_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY data_year), 2)             AS pct_of_year,
    ROUND(SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_null_passengers
FROM workspace.nyc_taxi.yellow_trips_raw
GROUP BY data_year, VendorID
ORDER BY data_year, VendorID;

### 📋 Vendor Analysis — Key Findings & Cleaning Implications

---

#### Vendor Landscape

| VendorID | Name | Status |
|---|---|---|
| 1 | Creative Mobile Technologies (CMT) | Keep — major vendor |
| 2 | VeriFone Inc. | Keep — dominant vendor |
| 4 | Unknown | Keep — low volume, clean data |
| 6 | Unknown Legacy | ❌ DROP — 100% null passenger_count every year |
| 5 | Unknown | ❌ DROP — 100% null, negligible volume |
| 7 | New Vendor (2024+) | Keep — clean data, growing |

---

#### Null Rate Trend — Industry-Wide Reporting Degradation

| Year | Vendor 1 Null % | Vendor 2 Null % |
|---|---|---|
| 2019 | 0.00% | 0.82% |
| 2022 | 2.56% | 3.60% |
| 2023 | 5.07% | 2.80% |
| 2024 | 8.90% | 10.25% |
| 2025 | 16.34% | 25.50% |

> ⚠️ This is NOT a data error — it is an industry-wide shift
> in how vendors report passenger_count and RatecodeID.
> Both major vendors are affected. Trips are real and valid.

---

#### Cleaning Rules Derived From This Section

1. **DROP** all trips where `VendorID IN (5, 6)` — unreliable vendors, negligible volume
2. **KEEP** all trips where `passenger_cou

### Section 4 — Financial Column Distributions

Now we look at the columns that directly drive our business question:
fare_amount, tip_amount, and total_amount.

We need to find:
- Minimum and maximum values — are there negatives? Extremes?
- Percentile distributions — where are the realistic boundaries?
- How average fare and tip have changed year over year

These distributions define our exact cleaning thresholds in `03_data_cleaning.sql`.
We do not guess thresholds — we let the data tell us.

In [0]:
SELECT
    data_year,
    COUNT(*)                                         AS total_trips,
    -- Fare
    ROUND(MIN(fare_amount), 2)                       AS min_fare,
    ROUND(PERCENTILE(fare_amount, 0.01), 2)          AS p1_fare,
    ROUND(AVG(fare_amount), 2)                       AS avg_fare,
    ROUND(PERCENTILE(fare_amount, 0.99), 2)          AS p99_fare,
    ROUND(MAX(fare_amount), 2)                       AS max_fare,
    -- Tip
    ROUND(MIN(tip_amount), 2)                        AS min_tip,
    ROUND(AVG(tip_amount), 2)                        AS avg_tip,
    ROUND(PERCENTILE(tip_amount, 0.99), 2)           AS p99_tip,
    ROUND(MAX(tip_amount), 2)                        AS max_tip,
    -- Total
    ROUND(MIN(total_amount), 2)                      AS min_total,
    ROUND(PERCENTILE(total_amount, 0.01), 2)         AS p1_total,
    ROUND(AVG(total_amount), 2)                      AS avg_total,
    ROUND(PERCENTILE(total_amount, 0.99), 2)         AS p99_total,
    ROUND(MAX(total_amount), 2)                      AS max_total
FROM workspace.nyc_taxi.yellow_trips_raw
WHERE VendorID NOT IN (5, 6)
GROUP BY data_year
ORDER BY data_year;


### 📋 Financial Distributions — Key Findings & Cleaning Implications

---

#### Outlier Summary

| Column | Issue | Example | Action |
|---|---|---|---|
| `fare_amount` | Negative values | -$133M in 2022 | Remove fare < $2.50 (NYC minimum) |
| `fare_amount` | Extreme positives | $943K in 2019 | Remove fare > $500 (p99.9 is ~$100) |
| `tip_amount` | Negative values | -$411 | Remove tip < $0 |
| `tip_amount` | Extreme positives | $133M in 2022 | Remove tip > $200 |
| `total_amount` | Negative values | -$2,567 | Remove total < $2.50 |
| `total_amount` | Extreme positives | $1.08M | Remove total > $600 |

---

#### Fare Trend — Meaningful Signal

| Year | Avg Fare | Avg Total | Interpretation |
|---|---|---|---|
| 2019 | $13.41 | $19.19 | Pre-COVID baseline |
| 2022 | $10.31 | $21.63 | Lower fare, higher surcharges post-COVID |
| 2023 | $19.52 | $28.46 | TLC rate increase took effect |
| 2024 | $19.27 | $27.83 | Stable |
| 2025 | $18.06 | $26.52 | Slight dip — congestion pricing impact? |

---

#### Cleaning Rules Derived From This Section

1. **REMOVE** trips where `fare_amount < 2.50` — below NYC minimum fare
2. **REMOVE** trips where `fare_amount > 500` — extreme outlier, not a real trip
3. **REMOVE** trips where `tip_amount < 0` — impossible
4. **REMOVE** trips where `tip_amount > 200` — extreme outlier
5. **REMOVE** trips where `total_amount < 2.50` — below minimum possible fare
6. **REMOVE** trips where `total_amount > 600` — extreme outlier

### Section 5 — Trip Distance and Duration Distributions

Distance and duration are critical for revenue per mile and revenue per minute —
the two metrics that tell a driver which trips are actually worth taking.

We check:
- Negative or zero distances — meter errors
- Extremely long distances — wrong year data or data entry errors
- Trip duration — calculated from pickup/dropoff timestamps
- Zero duration trips — meter left running or system errors

In [0]:
WITH base AS (
    SELECT
        data_year,
        trip_distance,
        ROUND((UNIX_TIMESTAMP(tpep_dropoff_datetime)
             - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60.0, 2) AS trip_minutes,
        YEAR(tpep_pickup_datetime) AS pickup_year
    FROM workspace.nyc_taxi.yellow_trips_raw
    WHERE VendorID NOT IN (5, 6)
)

SELECT
    data_year                                         AS year,
    ROUND(AVG(trip_distance), 2)                      AS avg_miles,
    ROUND(PERCENTILE(trip_distance, 0.99), 2)         AS max_realistic_miles,
    ROUND(MAX(trip_distance), 2)                      AS absolute_max_miles,
    ROUND(AVG(trip_minutes), 2)                       AS avg_mins,
    ROUND(PERCENTILE(trip_minutes, 0.99), 2)          AS max_realistic_mins,
    ROUND(MAX(trip_minutes), 2)                       AS absolute_max_mins,
    SUM(CASE WHEN pickup_year != data_year 
        THEN 1 ELSE 0 END)                            AS wrong_year_trips
FROM base
GROUP BY data_year
ORDER BY data_year;

### 📋 Distance & Duration — Key Findings & Cleaning Implications

---

#### What Normal Looks Like
99% of all trips across every year fall within:
- **Under 20 miles** — consistent across all years
- **Under 70 minutes** — consistent across all years

These are our realistic boundaries. Anything beyond these is an outlier.

---

#### Average Trip is Getting Longer Over Time

| Year | Avg Miles | Avg Minutes | Interpretation |
|---|---|---|---|
| 2019 | 3.02 mi | 18 mins | Short city hops, heavy traffic |
| 2022 | 5.96 mi | 53 mins | Post-COVID — longer suburban trips |
| 2023 | 4.09 mi | 17 mins | Back to normal city patterns |
| 2024 | 4.98 mi | 17 mins | Slightly longer trips |
| 2025 | 6.83 mi | 17 mins | Longest avg — congestion pricing pushing riders further out |

> ⚠️ 2022 average of 53 minutes is suspicious — likely pulled up
> by extreme outliers (max of 10.3M minutes = 20 years). 
> Confirms we need aggressive duration filtering.

---

#### Extreme Outliers — Clearly Bad Data

| Year | Absolute Max Miles | Absolute Max Minutes | Verdict |
|---|---|---|---|
| 2019 | 45,977 mi | 43,648 mins (30 days) | System error |
| 2022 | 389,678 mi | 10,322,055 mins (20 years) | System error |
| 2023 | 345,729 mi | 10,029 mins (7 days) | System error |
| 2024 | 398,608 mi | 9,767 mins (6.7 days) | System error |
| 2025 | 397,994 mi | 14,880 mins (10 days) | System error |

No taxi trip is 389,000 miles or 20 years long. These are meter malfunctions
or system timestamp errors — not real trips.

---

#### Wrong Year Trips — Manageable

| Year | Wrong Year Trips | % of Total |
|---|---|---|
| 2019 | 1,442 | 0.002% |
| 2022 | 561 | 0.001% |
| 2023 | 104 | 0.000% |
| 2024 | 56 | 0.000% |
| 2025 | 29 | 0.000% |

Tiny volume, but we remove them anyway — a 2018 trip
in the 2019 file has no business being in our analysis.

---

#### Cleaning Rules Derived From This Section

1. **REMOVE** trips where `trip_distance <= 0` — meter never started
2. **REMOVE** trips where `trip_distance > 100` — p99 is 20 miles, 100 is generous cap
3. **REMOVE** trips where `trip_minutes <= 0` — dropoff before or same as pickup
4. **REMOVE** trips where `trip_minutes > 300` — 5 hours is absolute maximum for a NYC taxi trip
5. **REMOVE** trips where `YEAR(tpep_pickup_datetime) != data_year` — wrong year trips

### Section 6 — Temporal Patterns

This is where the business question starts getting answered.

We look at trip volume and revenue by:
- Hour of day — which hours are most profitable?
- Day of week — weekdays vs weekends
- Month — seasonality patterns

This directly answers: "which hours should a driver prioritize?"

A high volume hour is not always the most profitable hour.
We care about revenue per trip, not just trip count.

In [0]:
WITH base AS (
    SELECT
        data_year,
        HOUR(tpep_pickup_datetime)  AS pickup_hour,
        total_amount,
        tip_amount,
        fare_amount,
        trip_distance
    FROM workspace.nyc_taxi.yellow_trips_raw
    WHERE VendorID NOT IN (5, 6)
      AND fare_amount BETWEEN 2.50 AND 500
      AND total_amount BETWEEN 2.50 AND 600
      AND trip_distance BETWEEN 0.1 AND 100
)

SELECT
    pickup_hour                             AS hour,
    COUNT(*)                                AS total_trips,
    ROUND(AVG(total_amount), 2)             AS avg_total,
    ROUND(AVG(tip_amount), 2)               AS avg_tip,
    ROUND(AVG(fare_amount), 2)              AS avg_fare,
    ROUND(AVG(trip_distance), 2)            AS avg_miles,
    ROUND(AVG(tip_amount / 
        NULLIF(fare_amount, 0)) * 100, 2)   AS avg_tip_pct
FROM base
GROUP BY pickup_hour
ORDER BY pickup_hour;

### 📋 Revenue by Hour — Observations

---

#### Trip Volume by Hour
- Volume is lowest between 2am–4am (1–3M trips)
- Volume builds from 7am, peaks at 5pm–6pm (15–16M trips)
- Late night 10pm–midnight sees moderate recovery (~10–13M trips)

---

#### Average Total Revenue by Hour
- Lowest average total: **2am ($21.60)**
- Highest average total: **5am ($30.09)**
- Midday 8am–noon clusters tightly between $22–$23
- Afternoon 1pm–4pm gradually rises from $24 to $25

---

#### Average Tip Percentage by Hour
- Lowest tip %: **4am (14.92%)** and **5am (15.35%)**
- Highest tip %: **6pm and 7pm (20.77%)**
- Evening hours 5pm–9pm consistently above 20%
- Overnight and early morning below 18%

---

#### Observations Worth Investigating in Revenue Analysis
- 4am–6am has highest avg total but lowest tip % — suggests longer/airport trips
- 5pm–7pm has highest tip % but not highest avg total — suggests short high-tip trips
- High volume hours (9am–11am) do not correspond to high avg revenue per trip
- These patterns warrant deeper zone-level investigation in `04_revenue_analysis.sql`

### Section 7 — Day of Week Patterns

We look at how trip volume and revenue vary across days of the week.

Key questions for EDA:
- Are weekends busier or quieter than weekdays?
- Does average revenue per trip differ by day?
- Are tip patterns different on weekends vs weekdays?

Note: 0 = Monday, 6 = Sunday (Spark default)

In [0]:
WITH base AS (
    SELECT
        DAYOFWEEK(tpep_pickup_datetime) AS day_of_week,
        DATE_FORMAT(tpep_pickup_datetime, 'EEEE')  AS day_name,
        total_amount,
        tip_amount,
        fare_amount,
        trip_distance
    FROM workspace.nyc_taxi.yellow_trips_raw
    WHERE VendorID NOT IN (5, 6)
      AND fare_amount BETWEEN 2.50 AND 500
      AND total_amount BETWEEN 2.50 AND 600
      AND trip_distance BETWEEN 0.1 AND 100
)

SELECT
    day_of_week,
    day_name,
    COUNT(*)                                          AS total_trips,
    ROUND(AVG(total_amount), 2)                       AS avg_total,
    ROUND(AVG(tip_amount), 2)                         AS avg_tip,
    ROUND(AVG(fare_amount), 2)                        AS avg_fare,
    ROUND(AVG(trip_distance), 2)                      AS avg_miles,
    ROUND(AVG(tip_amount /
        NULLIF(fare_amount, 0)) * 100, 2)             AS avg_tip_pct
FROM base
GROUP BY day_of_week, day_name
ORDER BY day_of_week;

### 📋 Day of Week — Observations

---

#### Trip Volume by Day
- Volume is relatively even across all days (30M–37M trips)
- Thursday is the busiest day (37M trips)
- Sunday and Monday are the quietest (30M trips each)
- Tuesday through Friday form the busy weekday core

---

#### Average Total Revenue by Day
- Highest avg total: **Monday ($24.81)**
- Lowest avg total: **Saturday ($22.89)**
- Weekdays (Mon–Thu) consistently above $24
- Weekend pattern: Friday dips slightly, Saturday is lowest of the week

---

#### Tip Percentage by Day — Clear Weekday vs Weekend Split

| Day Type | Days | Avg Tip % |
|---|---|---|
| Weekdays | Mon–Thu | 19.3%–19.7% |
| Transition | Friday | 19.1% |
| Weekend | Sat–Sun | 18.0%–18.1% |

Weekdays have noticeably higher tip percentages than weekends.
The drop from Friday to Saturday is the sharpest transition.

---

#### Average Miles by Day
- Sunday has the longest avg trip (3.74 miles)
- Tuesday–Wednesday have the shortest (3.21–3.24 miles)
- Weekend trips tend to be longer but generate less revenue per trip

---

#### Observations Worth Investigating in Revenue Analysis
- Saturday has lowest avg total AND lowest tip % — worth examining by zone
- Sunday has longest avg trips but lower tip % than weekdays
- Weekday mid-week (Tue–Thu) shows best balance of volume and tip %
- Weekend longer trips not translating to higher revenue warrants deeper look

### Section 8 — Zone Patterns

The most important dimension of our business question.
PULocationID = pickup zone (263 possible NYC taxi zones)

We look at:
- Which zones generate the highest avg revenue per trip
- Which zones have the highest trip volume
- Which zones have the best tip percentages

This is pure observation — recommendations come in `04_revenue_analysis.sql`

In [0]:
WITH base AS (
    SELECT
        PULocationID,
        total_amount,
        tip_amount,
        fare_amount,
        trip_distance
    FROM workspace.nyc_taxi.yellow_trips_raw
    WHERE VendorID NOT IN (5, 6)
      AND fare_amount BETWEEN 2.50 AND 500
      AND total_amount BETWEEN 2.50 AND 600
      AND trip_distance BETWEEN 0.1 AND 100
)

SELECT
    PULocationID                                        AS zone_id,
    COUNT(*)                                            AS total_trips,
    ROUND(AVG(total_amount), 2)                         AS avg_total,
    ROUND(AVG(tip_amount), 2)                           AS avg_tip,
    ROUND(AVG(fare_amount), 2)                          AS avg_fare,
    ROUND(AVG(trip_distance), 2)                        AS avg_miles,
    ROUND(AVG(tip_amount /
        NULLIF(fare_amount, 0)) * 100, 2)               AS avg_tip_pct
FROM base
GROUP BY PULocationID
ORDER BY avg_total DESC
LIMIT 30;

### 📋 Zone Patterns — Observations

---

#### Important Context
Zone IDs alone are not meaningful without names.
Key zones we can identify from TLC documentation:

| Zone ID | Name | Type |
|---|---|---|
| 132 | JFK Airport | Airport |
| 138 | LaGuardia Airport | Airport |
| 1 | Newark Airport | Airport |
| 132 | JFK Airport | Airport |
| 70 | East Harlem North | Neighborhood |
| 265 | Unknown | Unknown |

---

#### Top Zones by Avg Total Revenue

- **Zone 1 (Newark):** Highest avg total ($100.83) — long distance airport runs
- **Zone 5:** Second highest ($88.38) but only 1,053 trips — low volume, unreliable average
- **Zone 265:*